# Ordered Logistic Regression Results: Exploration with `mlcroissant`
This notebook demonstrates how to use the [`mlcroissant`](https://github.com/mlcommons/croissant) library to load, explore, and process the **Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya** dataset, defined by its Croissant schema.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load the dataset metadata and prepare for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset from the Croissant schema
dataset = mlc.Dataset(croissant_url)

# Access and print metadata
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, their `@id`s, fields, and columns. This helps guide targeted data extraction.

Let's enumerate available record sets and then inspect their fields and columns, referencing all by their Croissant `@id`.

In [ ]:
# List all record sets and their fields (@id, name, columns) using Croissant API.
def print_record_sets(ds):
    print("Available record sets:\n----------------------")
    for rs in ds.record_sets:
        print(f"- @id: {rs.id}\n  name: {rs.name}")
        if hasattr(rs, 'fields'):
            print("  Fields:")
            for f in rs.fields:
                print(f"    - @id: {f.id} | name: {f.name} | dataType: {getattr(f, 'data_type', None)} | column: {getattr(f, 'column', None)}")
        if hasattr(rs, 'columns'):
            print("  Columns:")
            for c in rs.columns:
                print(f"    - @id: {c.id} | name: {c.name}")
        print()

print_record_sets(dataset)

To preview actual records, pick a record set of interest and print its first few records:

*(Replace `<record_set_id>` below with an actual `@id` from the dataset's record sets as shown in the previous output.)*

In [ ]:
# Example: Print first 3 records from a record set
# Replace '<record_set_id>' with a valid record set @id from above (e.g., cr:RecordSet/Results)
selected_record_set = None
if len(dataset.record_sets):
    selected_record_set = dataset.record_sets[0].id  # pick the first one for demonstration
    print(f"Showing records from: {selected_record_set}")
    for i, rec in enumerate(dataset.records(record_set=selected_record_set)):
        print(rec)
        if i > 2:
            break
else:
    print("No record sets in the dataset.")


## 3. Data Extraction
Extract data from one or more record sets into pandas DataFrames for further processing and analysis.

All lookups and extraction reference entities by their Croissant `@id`.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

# Extract each record set into a pandas DataFrame
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set '{record_set_id}', shape = {df.shape}")
    else:
        print(f"No records for record set '{record_set_id}'.")

# Show DataFrame columns for the first available record set
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in record set '{first_rs}':")
    print(dataframes[first_rs].columns.tolist())
    dataframes[first_rs].head()
else:
    print("No DataFrames loaded.")


## 4. Exploratory Data Analysis (EDA)
Conduct EDA with typical steps such as filtering, normalizing, and grouping.

*You must use Croissant `@id` for all field/column references.*

In [ ]:
import numpy as np

# Identify a numeric field/column from one of the loaded DataFrames.
# This example will pick one automatically if it exists.
analysis_rs_id = None
numeric_field_id = None
group_field_id = None
for rs_id, df in dataframes.items():
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        analysis_rs_id = rs_id
        numeric_field_id = num_cols[0]  # Select first as example
        # Select a groupable field (categorical/object)
        group_candidates = df.select_dtypes(include=['object', 'category']).columns.tolist()
        if group_candidates:
            group_field_id = group_candidates[0]
        break

if analysis_rs_id and numeric_field_id:
    print(f"Analysis will use record set: {analysis_rs_id}\nNumeric field: {numeric_field_id}\nGroup field: {group_field_id}")
    # Remove NaNs and filter for values above a threshold (e.g., 10 or minimum+epsilon)
    threshold = 10
    df = dataframes[analysis_rs_id]
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    col_norm = f"{numeric_field_id}_normalized"
    filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized " + numeric_field_id + " for filtered records:")
    print(filtered_df[[numeric_field_id, col_norm]].head())

    # Group by key field and compute means
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id}, mean of {numeric_field_id}:")
        print(grouped_df.head())
else:
    print("No numeric field or suitable record set available for EDA (check schema).")


## 5. Visualization
Visualize data distributions or relationships between selected fields using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field (if found above)
if analysis_rs_id and numeric_field_id:
    plt.figure(figsize=(7,4))
    sns.histplot(data=dataframes[analysis_rs_id], x=numeric_field_id, kde=True)
    plt.title(f"Distribution of {numeric_field_id} in record set {analysis_rs_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping info is available, show aggregate mean plot
    if group_field_id and group_field_id in dataframes[analysis_rs_id].columns:
        plt.figure(figsize=(8,4))
        grouped = dataframes[analysis_rs_id].groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean of {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean of {numeric_field_id}")
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No analyzable numeric/group fields found.")


## 6. Conclusion
With the `mlcroissant` library you can load and interpret machine-readable FAIR datasets directly, using only the Croissant schema URL. Fields, record sets, and columns can be manipulated by referencing their Croissant `@id`s from metadata, making your workflow robust and reproducible.

In this notebook, we loaded the metadata and records, identified available data entities (record sets, fields), extracted records to DataFrames, demonstrated basic EDA & transformation, and visualized distributions. You can easily adapt these steps for in-depth modeling or knowledge graph applications.